In [20]:
import pandas as pd
import numpy as np
from sklearn.linear_model import LinearRegression, Ridge, Lasso, RidgeCV
from sklearn.metrics import mean_squared_error, r2_score, mean_absolute_error
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import PolynomialFeatures, StandardScaler, OneHotEncoder
from sklearn.model_selection import GridSearchCV, RandomizedSearchCV, train_test_split, TimeSeriesSplit
from sklearn.compose import ColumnTransformer
from xgboost import XGBRegressor, callback

In [21]:
def load_sheet(file, sheet_name):
    df = pd.read_excel("/Users/timmyg/Downloads/" + file, sheet_name=sheet_name, header=2)
    return df


def load_sheet_with_drop(file, sheet_name):
    df = pd.read_excel("/Users/timmyg/Downloads/" + file, sheet_name=sheet_name, header=2)
    df = df.iloc[1:].reset_index(drop=True)
    return df

In [22]:
#loading and reformatting all sheets from co2_source dataset
df_total_source = load_sheet_with_drop("co2_source.xlsx", "Total")
df_coal = load_sheet_with_drop("co2_source.xlsx", "Coal")
df_petroleum = load_sheet_with_drop("co2_source.xlsx", "Petroleum")
df_natural_gas = load_sheet_with_drop("co2_source.xlsx", "Natural gas")

In [23]:
#loading data from co2_sector dataset
df_total_sector = load_sheet("co2_sector.xlsx", "Total")
df_residential = load_sheet("co2_sector.xlsx", "Residential")
df_commercial = load_sheet("co2_sector.xlsx", "Commercial")
df_industrial = load_sheet("co2_sector.xlsx", "Industrial")
df_transportation = load_sheet("co2_sector.xlsx", "Transportation")
df_electric_power = load_sheet("co2_sector.xlsx", "Electric power")

In [24]:
#loading data from Electric Vehicle sheet
df_EV = pd.read_excel("/Users/timmyg/Downloads/EV_Registrations.xlsx",header=2)

In [25]:
#df_co2_total = pd.read_excel("CO2_total.xlsx", sheet_name="Total CO2",header=2) --> Same as df_total
df_co2_capita = load_sheet("CO2_total.xlsx", "Per capita")
df_co2_per_billion_btu = load_sheet("CO2_total.xlsx", "CO2 per billion Btu")
df_co2_per_million_dollars = load_sheet("CO2_total.xlsx", "CO2 per million dollars")

drop_threshold = 0.75 * len(df_co2_per_million_dollars)
df_co2_per_million_dollars = df_co2_per_million_dollars.dropna(axis=1, thresh=len(df_co2_per_million_dollars) - drop_threshold)

In [26]:
#loading and reformatting all sheets from indicator_other dataset
df_total_population = load_sheet("indicator_other.xlsx", "Total population")
df_real_gdp = load_sheet("indicator_other.xlsx", "Real GDP")
df_current_dollar_gdp = load_sheet("indicator_other.xlsx", "Current-dollar GDP")
df_hdd = load_sheet("indicator_other.xlsx", "HDD")
df_cdd = load_sheet("indicator_other.xlsx", "CDD")

In [27]:
df_total_source
new_df = pd.DataFrame(df_total_source.values[0:], columns=df_total_source.iloc[0])
new_df = new_df.dropna()
new_df = new_df.T
new_df = pd.DataFrame(new_df.values[1:], columns=new_df.iloc[0])
new_df = new_df.reset_index()
new_df

AL,index,AL,AR,AZ,CA,CO,CT,DC,DE,FL,...,TN,TX,UT,VA,VT,WA,WI,WV,WY,US
0,0,62.4,21.3,15.6,197.5,27.2,36.9,7.7,11.3,52.4,...,58.4,240.5,21.0,64.7,3.6,30.1,60.3,48.0,9.2,2914.6
1,1,61.1,21.8,16.9,208.5,29.8,37.9,7.2,10.9,55.3,...,57.9,241.1,20.5,64.8,3.9,31.1,62.0,52.1,10.4,2942.8
2,2,66.3,23.3,18.4,209.5,30.2,39.2,7.2,11.8,62.2,...,58.6,252.0,20.1,67.8,3.9,32.4,64.3,52.5,11.6,3065.1
3,3,69.1,24.8,19.3,219.4,30.4,40.0,7.0,12.6,64.7,...,63.7,263.6,19.9,71.4,4.0,33.2,66.4,57.5,11.6,3186.0
4,4,71.8,26.8,19.9,238.3,32.1,41.3,7.7,12.6,68.0,...,59.9,270.1,21.2,73.4,3.7,35.0,68.9,60.6,11.7,3320.9
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
59,59,106.0,64.9,92.5,354.6,91.3,36.2,2.8,13.6,232.9,...,91.9,681.7,61.4,106.4,6.0,84.0,94.6,85.3,59.0,5133.7
60,60,98.1,54.5,80.1,299.6,79.5,33.4,2.4,12.2,207.0,...,82.9,625.5,57.3,97.8,5.4,68.2,86.8,76.7,55.5,4571.8
61,61,108.4,61.8,82.8,321.9,84.8,36.2,2.6,12.7,225.6,...,92.2,662.8,62.1,98.1,5.5,73.5,92.4,88.2,54.5,4891.3
62,62,109.5,63.3,80.4,324.1,86.5,37.0,2.6,12.7,231.4,...,91.3,666.1,60.2,96.4,5.5,74.9,90.8,79.1,56.3,4931.1


In [30]:
train = new_df.sample(frac=0.7)
test = new_df[~new_df.index.isin(train.index)]
ind = ["index"]
dep = ["US"]

In [32]:
train["US"] = train["US"].astype("category")
params = {
    "n_estimators": [300, 600, 1000],
    "learning_rate": [0.01, 0.05, 0.1],
    "max_depth": [3, 5, 7],
    "subsample": [0.7, 0.9, 1.0],
    "colsample_bytree": [0.7, 0.9, 1.0],
    "min_child_weight": [1, 3, 5],
    "gamma": [0, 0.1, 0.3],
    "reg_lambda": [1, 1.5, 2],
}

xgb = XGBRegressor(random_state=42, n_jobs=-1, tree_method="hist", enable_categorical=True)
search = RandomizedSearchCV(xgb, params, n_iter=25, scoring="r2", cv=TimeSeriesSplit(n_splits=3), verbose=2, n_jobs=-1)
search.fit(train[ind], train["US"])
print("Best params:", search.best_params_)
print("Best R²:", search.best_score_)

Fitting 3 folds for each of 25 candidates, totalling 75 fits
[CV] END colsample_bytree=0.7, gamma=0.3, learning_rate=0.01, max_depth=7, min_child_weight=1, n_estimators=300, reg_lambda=1, subsample=0.9; total time=   0.0s
[CV] END colsample_bytree=0.9, gamma=0, learning_rate=0.01, max_depth=7, min_child_weight=5, n_estimators=1000, reg_lambda=1, subsample=1.0; total time=   0.1s
[CV] END colsample_bytree=0.9, gamma=0, learning_rate=0.01, max_depth=7, min_child_weight=5, n_estimators=1000, reg_lambda=1, subsample=1.0; total time=   0.1s
[CV] END colsample_bytree=0.9, gamma=0.1, learning_rate=0.01, max_depth=3, min_child_weight=1, n_estimators=300, reg_lambda=2, subsample=1.0; total time=   0.0s
[CV] END colsample_bytree=0.9, gamma=0.1, learning_rate=0.01, max_depth=3, min_child_weight=1, n_estimators=300, reg_lambda=2, subsample=1.0; total time=   0.0s
[CV] END colsample_bytree=0.7, gamma=0.3, learning_rate=0.01, max_depth=7, min_child_weight=1, n_estimators=300, reg_lambda=1, subsample